# Used Car Price Prediction

### Problem statement.

- This dataset comprises used cars sold on cardehko.com in India as well as important features of these cars.
- If user can predict the price of the car based on input features.
- Prediction results can be used to give new seller the price suggestion based on market condition.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn as skl
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

In [2]:
data = pd.read_csv(
    "S:/AI ML/Machine Learning/Datasets/cardekho_imputated.csv", index_col=[0]
)

In [3]:
data.head()

,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


In [4]:
# Data Cleaning
data.isnull().sum()

car_name             0
brand                0
model                0
vehicle_age          0
km_driven            0
seller_type          0
fuel_type            0
transmission_type    0
mileage              0
engine               0
max_power            0
seats                0
selling_price        0
dtype: int64

In [5]:
# Dropping unnecessary columns
data = data.drop(columns=["car_name", "brand"])

In [6]:
# Different Types of Features

num_features = [feature for feature in data.select_dtypes(include="number").columns]
print(f"Numerical Features : {len(num_features)}\n{num_features}")

cat_features = [
    feature for feature in data.select_dtypes(include=["object", "category"]).columns
]
print(f"\nCategorical Features : {len(cat_features)}\n{cat_features}")

desc_features = [
    feature for feature in num_features if len(data[feature].unique()) <= 25
]
print(f"\nDiscrete Features : {len(desc_features)}\n{desc_features}")

cont_features = [
    feature for feature in num_features if len(data[feature].unique()) > 25
]
print(f"\nContinuous Features : {len(cont_features)}\n{cont_features}")

Numerical Features : 7
['vehicle_age', 'km_driven', 'mileage', 'engine', 'max_power', 'seats', 'selling_price']

Categorical Features : 4
['model', 'seller_type', 'fuel_type', 'transmission_type']

Discrete Features : 2
['vehicle_age', 'seats']

Continuous Features : 5
['km_driven', 'mileage', 'engine', 'max_power', 'selling_price']


In [7]:
# Independent and Dependent Features
X = data.drop("selling_price", axis=1)
y = data["selling_price"]

In [8]:
# Feature Encoding and Scaling
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
X["model"] = le.fit_transform(X["model"])

In [9]:
# One hot encoding three columns
oh_columns = ["seller_type", "fuel_type", "transmission_type"]
num_features = X.select_dtypes(exclude="object").columns

oh_transformer = skl.preprocessing.OneHotEncoder(drop="first")
num_transformer = skl.preprocessing.StandardScaler()
preprocessor = skl.compose.ColumnTransformer(
    [
        ("OneHotEncoder", oh_transformer, oh_columns),
        ("StandardScaler", num_transformer, num_features),
    ],
    remainder="passthrough",
)

In [10]:
X = preprocessor.fit_transform(X)

In [11]:
# Train Test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=43
)

In [12]:
# Model Selection
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor

In [13]:
from sklearn.metrics import mean_squared_error, r2_score, root_mean_squared_error

In [15]:
models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso(),
    "DecisionTreeRegressor": DecisionTreeRegressor(),
    "KNeighborsRegressor": KNeighborsRegressor(),
    "RandomForestRegressor": RandomForestRegressor(),
    "AdaBoostRegressor": AdaBoostRegressor(),
    "GradientBoostRegressor": GradientBoostingRegressor(),
    "XGBoostRegressor": XGBRegressor(),
}

In [16]:
for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train, y_train)

    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    print(f"Model : {list(models.keys())[i]}")
    print("Train Set")
    print(f"R2 Score : {r2_score(y_train, y_pred_train)}")
    print(f"Mean Squared Error : {mean_squared_error(y_train, y_pred_train)}")
    print(f"Root Mean Squared Error : {root_mean_squared_error(y_train, y_pred_train)}\n")
    print("Test Set")
    print(f"R2 Score : {r2_score(y_test, y_pred_test)}")
    print(f"Mean Squared Error : {mean_squared_error(y_test, y_pred_test)}")
    print(f"Root Mean Squared Error : {root_mean_squared_error(y_test, y_pred_test)}")
    print("--------------------------------")

Model : LinearRegression
Train Set
R2 Score : 0.62617406181362
Mean Squared Error : 288921093492.94214
Root Mean Squared Error : 537513.8077230594

Test Set
R2 Score : 0.6407855689138455
Mean Squared Error : 325279362843.61224
Root Mean Squared Error : 570332.6773415777
--------------------------------
Model : Ridge
Train Set
R2 Score : 0.6261736910144444
Mean Squared Error : 288921380074.72345
Root Mean Squared Error : 537514.0743038488

Test Set
R2 Score : 0.6407786147362571
Mean Squared Error : 325285660058.47705
Root Mean Squared Error : 570338.1979654502
--------------------------------
Model : Lasso
Train Set
R2 Score : 0.6261740554448336
Mean Squared Error : 288921098415.22424
Root Mean Squared Error : 537513.8123018089

Test Set
R2 Score : 0.6407835416791111
Mean Squared Error : 325281198565.02454
Root Mean Squared Error : 570334.2866819638
--------------------------------
Model : DecisionTreeRegressor
Train Set
R2 Score : 0.9994902032404859
Mean Squared Error : 394009677.15768

In [17]:
# Parameters of good models

rf_params = {
    "n_estimators": [100, 200, 300, 400, 500, 800, 1000],
    "max_depth": [None, 5, 8, 10, 20],
    "min_samples_split": [2, 5, 10, 15, 20],
    "max_features": ["auto", 5, 7, 9, 8],
}
xb_params = {
    "n_estimators": [100, 200, 300, 500, 800],
    "max_depth": [3, 4, 5, 6, 8, 10, 12, 20, 30],
    "learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2],
    "colsample_bytree": [0.3, 0.4, 0.5, 0.7, 0.8, 0.9, 1.0],
}

In [18]:
# Models list for Hyperperparameter Tuning
randomcv_models = [
    ("RandomForest", RandomForestRegressor(), rf_params),
    ("XGBoostRegressor", XGBRegressor(), xb_params),
]

In [19]:
# Tuning the models
from sklearn.model_selection import RandomizedSearchCV

for name, model, params in randomcv_models:
    random = RandomizedSearchCV(
        estimator=model,
        param_distributions=params,
        n_iter=100,
        cv=3,
        scoring="r2",
        n_jobs=-1,
    )

    random.fit(X_train, y_train)

    y_pred_train = random.predict(X_train)
    y_pred_test = random.predict(X_test)

    print(f"Model : {name}")
    print(f"Best Params : {random.best_params_}")
    print("Train Set")
    print(f"R2 Score : {r2_score(y_train, y_pred_train)}")
    print(f"Mean Squared Error : {mean_squared_error(y_train, y_pred_train)}")
    print(
        f"Root Mean Squared Error : {root_mean_squared_error(y_train, y_pred_train)}\n"
    )
    print("Test Set")
    print(f"R2 Score : {r2_score(y_test, y_pred_test)}")
    print(f"Mean Squared Error : {mean_squared_error(y_test, y_pred_test)}")
    print(f"Root Mean Squared Error : {root_mean_squared_error(y_test, y_pred_test)}")
    print("\n--------------------------------\n")

Model : RandomForest
Best Params : {'n_estimators': 100, 'min_samples_split': 5, 'max_features': 8, 'max_depth': 20}
Train Set
R2 Score : 0.9698042448527956
Mean Squared Error : 23337574268.658104
Root Mean Squared Error : 152766.40425387418

Test Set
R2 Score : 0.8887334325942206
Mean Squared Error : 100755189712.48477
Root Mean Squared Error : 317419.5799135346

--------------------------------

Model : XGBoostRegressor
Best Params : {'n_estimators': 300, 'max_depth': 4, 'learning_rate': 0.2, 'colsample_bytree': 0.7}
Train Set
R2 Score : 0.9833874106407166
Mean Squared Error : 12839483392.0
Root Mean Squared Error : 113311.4453125

Test Set
R2 Score : 0.7928045988082886
Mean Squared Error : 187621621760.0
Root Mean Squared Error : 433153.125

--------------------------------

